# 01 - Bronze Ingestion

Load the two source CSVs into immutable Bronze Delta tables.

- Explicit schemas (no `inferSchema`)
- Add `_ingested_at` and `_source_file`
- Write Delta tables under `crop_medallion_architecture.bronze`
- Report row/column counts (no hardcoded expectations)

## 1. Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
)

from configs.project_config import PATHS, BRONZE_TABLES, SOURCE_FILES

print(f'Crop CSV    : {PATHS.crop_csv}')
print(f'Weather CSV : {PATHS.weather_csv}')
print(f'Bronze crop : {BRONZE_TABLES['crop_production_raw']}')
print(f'Bronze weather : {BRONZE_TABLES['weather_raw']}')

## 2. Explicit schemas

Defined from the data dictionary. Bronze keeps source values as-is;
type casts happen here only so Delta storage is typed.

In [0]:
crop_schema = StructType([
    StructField('province', StringType(), True),
    StructField('year', IntegerType(), True),
    StructField('quarter', StringType(), True),
    StructField('temperature_2m_mean', DoubleType(), True),
    StructField('temperature_2m_max', DoubleType(), True),
    StructField('temperature_2m_min', DoubleType(), True),
    StructField('precipitation_sum', DoubleType(), True),
    StructField('rain_sum', DoubleType(), True),
    StructField('precipitation_hours', DoubleType(), True),
    StructField('sunshine_duration', DoubleType(), True),
    StructField('shortwave_radiation_sum', DoubleType(), True),
    StructField('reference_evapotranspiration_mm', DoubleType(), True),
    StructField('wind_speed_10m_max', DoubleType(), True),
    StructField('wind_gusts_10m_max', DoubleType(), True),
    StructField('rain_normal', DoubleType(), True),
    StructField('temp_normal', DoubleType(), True),
    StructField('rainfall_deviation_pct', DoubleType(), True),
    StructField('temperature_anomaly_c', DoubleType(), True),
    StructField('oni_index', DoubleType(), True),
    StructField('crop', StringType(), True),
    StructField('production', DoubleType(), True),
    StructField('quarter_num', IntegerType(), True),
    StructField('time_index', IntegerType(), True),
    StructField('production_lag_1', DoubleType(), True),
    StructField('production_lag_2', DoubleType(), True),
    StructField('production_lag_4', DoubleType(), True),
    StructField('temp_anomaly_rolling', DoubleType(), True),
    StructField('rain_anomaly_rolling', DoubleType(), True),
    StructField('enso_phase_la_nina', BooleanType(), True),
    StructField('enso_phase_neutral', BooleanType(), True),
    StructField('crop_group', StringType(), True),
    StructField('growth_rate_yoy', DoubleType(), True),
    StructField('climate_stress', DoubleType(), True),
    StructField('risk_label', StringType(), True),
])

weather_schema = StructType([
    StructField('province', StringType(), True),
    StructField('year', IntegerType(), True),
    StructField('year_quarter', StringType(), True),
    StructField('temperature_2m_mean', DoubleType(), True),
    StructField('temperature_2m_max', DoubleType(), True),
    StructField('temperature_2m_min', DoubleType(), True),
    StructField('precipitation_sum', DoubleType(), True),
    StructField('rain_sum', DoubleType(), True),
    StructField('precipitation_hours', DoubleType(), True),
    StructField('sunshine_duration', DoubleType(), True),
    StructField('shortwave_radiation_sum', DoubleType(), True),
    StructField('reference_evapotranspiration_mm', DoubleType(), True),
    StructField('wind_speed_10m_max', DoubleType(), True),
    StructField('wind_gusts_10m_max', DoubleType(), True),
    StructField('quarter', StringType(), True),
    StructField('rain_normal', DoubleType(), True),
    StructField('temp_normal', DoubleType(), True),
    StructField('rainfall_deviation_pct', DoubleType(), True),
    StructField('temperature_anomaly_c', DoubleType(), True),
    StructField('oni_index', DoubleType(), True),
    StructField('enso_phase', StringType(), True),
])

## 3. Read CSVs and add ingestion metadata

In [0]:
def ingest_csv(path: str, schema: StructType, source_file: str):
    df = (
        spark.read
        .option('header', True)
        .schema(schema)
        .csv(path)
        .withColumn('_ingested_at', F.current_timestamp())
        .withColumn('_source_file', F.lit(source_file))
    )
    return df

crop_raw = ingest_csv(PATHS.crop_csv, crop_schema, SOURCE_FILES['crop_trend'])
weather_raw = ingest_csv(PATHS.weather_csv, weather_schema, SOURCE_FILES['weather'])

print('Crop sample:')
crop_raw.show(3, truncate=False)
print('Weather sample:')
weather_raw.show(3, truncate=False)

## 4. Write Bronze Delta tables

In [0]:
crop_table = BRONZE_TABLES['crop_production_raw']
weather_table = BRONZE_TABLES['weather_raw']

(
    crop_raw.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(crop_table)
)
print(f'Wrote: {crop_table}')

(
    weather_raw.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(weather_table)
)
print(f'Wrote: {weather_table}')

## 5. Validation

In [0]:
crop_df = spark.table(crop_table)
weather_df = spark.table(weather_table)

print('crop_production_raw')
print(f'  rows : {crop_df.count()}')
print(f'  cols : {len(crop_df.columns)}')
print(f'  columns: {crop_df.columns}')
print()
print('weather_raw')
print(f'  rows : {weather_df.count()}')
print(f'  cols : {len(weather_df.columns)}')
print(f'  columns: {weather_df.columns}')
print()

# Metadata columns present?
for name, df in [('crop', crop_df), ('weather', weather_df)]:
    has_ts = '_ingested_at' in df.columns
    has_file = '_source_file' in df.columns
    print(f'{name}: ingested_at={has_ts}, source_file={has_file}')

print()
print('Bronze ingestion complete.')
print('Next: Silver (02_silver/01_clean_crop_production)')